# Meta Feature Analysis - Part 1

# Data Loading

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_dir = notebook_dir.parent
repo_root = project_dir.parent

sys.path.insert(0, str(project_dir / "src"))
sys.path.insert(0, str(repo_root / "tabarena" / "tabarena"))

In [2]:
CONFIG = "config_1.yaml"

In [3]:
from dataclasses import replace

from mfa import load_config

config_path = project_dir / "configs" / CONFIG
config = load_config(config_path)

# Read the configured value from YAML
yaml_n_jobs = config.parallelism.n_jobs

# Keep this notebook safe: allow only 1..2 workers for this run
run_n_jobs = min(max(yaml_n_jobs, 1), 2)
effective_n_jobs = 2

# Build a temporary config override for this run only (does not edit YAML)
run_config = replace(
    config,
    parallelism=replace(config.parallelism, n_jobs=effective_n_jobs),
)

if yaml_n_jobs != run_n_jobs:
    print(
        f"Loaded {config_path.name}; parallelism.n_jobs={yaml_n_jobs}. "
        f"Temporarily using n_jobs={run_n_jobs} for this run."
    )
else:
    print(
        f"Loaded {config_path.name}; parallelism.n_jobs={yaml_n_jobs}. "
        "Proceeding with the analysis."
    )

Loaded config_1.yaml; parallelism.n_jobs=-1. Temporarily using n_jobs=1 for this run.


In [4]:
from mfa import run_analysis
from tabarena.nips2025_utils.fetch_metadata import load_task_metadata

result = run_analysis(run_config)

task_type_lookup = (
    load_task_metadata()[["dataset", "problem_type"]]
    .rename(columns={"problem_type": "task_type"})
    .drop_duplicates(subset="dataset")
)
result.metafeature_table = result.metafeature_table.merge(
    task_type_lookup, on="dataset", how="left", validate="many_to_one"
)
result.analysis_table = result.analysis_table.merge(
    task_type_lookup, on="dataset", how="left", validate="many_to_one"
)
result.analysis_table.head()

11:57:45 INFO mfa.pipeline: Starting analysis: comparisons=non_foundational_vs_foundational; scope=all benchmark datasets; unit=dataset; method_variant=default,tuned; n_jobs=2
11:57:45 INFO mfa.pipeline: Stage 1/5 raw results: cache hit (34110 rows, 51 dataset(s))
11:57:45 INFO mfa.pipeline: Stage 2/5 meta-features: trace enabled; metafeature caches remain active, so live per-split diagnostics appear only on cache misses
11:57:45 INFO mfa.pipeline: Stage 2/5 meta-features: pymfe enabled; rebuilding from split cache and reusing cached pymfe failures/incomplete outputs as-is
11:57:45 INFO mfa.pipeline: Stage 2/5 meta-features: building for all benchmark datasets
11:57:45 INFO mfa.metafeatures: Meta-features: preparing 51 dataset(s) with feature_sets=basic,redundancy,pymfe (n_jobs=2)
11:57:45 INFO mfa.metafeatures: Meta-features: trace enabled; split cache remains active, so live timing and warning diagnostics appear only on cache misses
11:57:45 INFO mfa.metafeatures: Meta-features: trac

Meta-features: 100%|██████████| 816/816 [00:37<00:00, 21.94split/s]
11:58:23 INFO mfa.metafeatures: Meta-features: complete in 00:00:38 (816 rows; 816 cached split(s), 0 repaired split(s), 0 recomputed split(s), 0 unrepaired pymfe output(s))
11:58:23 INFO mfa.pipeline: Stage 2/5 meta-features: ready (816 rows, 51 dataset(s))
11:58:24 INFO mfa.pipeline: Stage 3/5 pairwise gaps: cache hit (816 rows, 51 dataset(s))
11:58:24 INFO mfa.pipeline: Stage 4/5 analysis table: joining and aggregating at dataset level
/work/mherre/tabular-meta-feature-analysis/meta-feature-analysis/src/mfa/aggregation.py:150: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset_level = analysis.groupby(GROUP_COLUMNS, as_index=False, dropna=False).agg(**aggregations)
/work/mherre/tabula

,dataset,comparison_name,group_a_name,group_b_name,group_a_label,group_b_label,expected_direction,n_splits,n,d,...,best_a_norm_error,best_b_error,best_b_norm_error,delta_raw,delta_raw_std,delta_raw_sem,delta_norm,delta_norm_std,delta_norm_sem,task_type
0,APSFailure,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,50666.666667,170.0,...,0.832294,0.007284,0.333333,0.001366,0.002011,0.000670,0.498960,0.549148,0.183049,binary
1,Amazon_employee_access,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,21846.000000,9.0,...,0.001416,0.146592,0.778336,-0.029811,0.007122,0.002374,-0.776919,0.115570,0.038523,binary
2,Another-Dataset-on-used-Fiat-500,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,30,1025.333333,7.0,...,0.502696,719.852165,0.203675,12.029602,16.598990,3.030547,0.299021,0.414145,0.075612,regression
3,Bank_Customer_Churn,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,6666.666667,10.0,...,0.479097,0.125676,0.037007,0.004284,0.001790,0.000597,0.442090,0.159837,0.053279,binary
4,Bioresponse,non_foundational_vs_foundational,non-foundational,foundational,standard,transformer-based,NaN,9,2500.666667,1776.0,...,0.441583,0.123178,0.216194,0.003073,0.005005,0.001668,0.225389,0.360046,0.120015,binary


# Preprocessing

In [5]:
import pandas as pd

# -- Inspect what the result object contains --
print(f"config_hash:        {result.config_hash}")
print(f"comparison_name:    {result.comparison_name}")
print(f"analysis_table:     {result.analysis_table.shape}")
print(f"gap_table:          {result.gap_table.shape}")
print(f"metafeature_table:  {result.metafeature_table.shape}")
print(f"correlation_results: {len(result.correlation_results)} features tested")
print(
    f"correction_result:  {result.correction_result.method if result.correction_result else None}"
)
print(f"multivariate_result: {result.multivariate_result}")

config_hash:        c4472762a9293af4
comparison_name:    non_foundational_vs_foundational
analysis_table:     (51, 1133)
gap_table:          (816, 17)
metafeature_table:  (816, 1118)
correlation_results: 1114 features tested
correction_result:  bh
multivariate_result: None


## Build Task-Aware Analysis Tables

We construct two analysis matrices because the available meta-features depend on the supervised task type. The general matrix contains only features that are meaningful for both regression and classification tasks. The classification matrix is restricted to classification tasks and augments the general feature set with class-label-dependent features such as class counts, entropy, imbalance, and Pymfe classification-only descriptors.

The two matrices are analyzed separately downstream, so their preprocessing filters are estimated within their own task populations rather than forcing the classification-specific analysis to inherit decisions from the full regression-plus-classification table.

In [ ]:
import numpy as np

from mfa.metafeatures.basic import (
    CLASSIFICATION_PROBLEM_TYPES as BASIC_CLASSIFICATION_PROBLEM_TYPES,
)
from mfa.metafeatures.pymfe_catalog import PYMFE_CLASSIFICATION_ONLY
from mfa.stats.correlation import EXCLUDED_PREDICTOR_COLUMNS

BASIC_CLASSIFICATION_ONLY_FEATURES = frozenset(
    {
        "n_classes",
        "class_entropy",
        "majority_class_fraction",
        "minority_class_fraction",
        "class_imbalance_ratio",
    }
)
CLASSIFICATION_PROBLEM_TYPES = set(BASIC_CLASSIFICATION_PROBLEM_TYPES)

ANALYSIS_CONTEXT_COLUMNS = [
    "dataset",
    "task_type",
    "comparison_name",
    "group_a_name",
    "group_b_name",
    "group_a_label",
    "group_b_label",
    "expected_direction",
    "n_splits",
    "best_a_error",
    "best_a_norm_error",
    "best_b_error",
    "best_b_norm_error",
    "delta_raw",
    "delta_raw_std",
    "delta_raw_sem",
    "delta_norm",
    "delta_norm_std",
    "delta_norm_sem",
]


def _is_pymfe_classification_only(column: str) -> bool:
    if not column.startswith("pymfe__"):
        return False
    raw_feature = column.removeprefix("pymfe__").split(".", maxsplit=1)[0]
    return raw_feature in PYMFE_CLASSIFICATION_ONLY


def is_classification_only_feature(column: str) -> bool:
    return (
        column in BASIC_CLASSIFICATION_ONLY_FEATURES
        or _is_pymfe_classification_only(column)
    )


def infer_numeric_predictors(
    table: pd.DataFrame, *, target: str = "delta_norm"
) -> list[str]:
    predictors = []
    for column in table.columns:
        if (
            column == target
            or column in EXCLUDED_PREDICTOR_COLUMNS
            or column.startswith("best_")
        ):
            continue
        numeric_values = pd.to_numeric(table[column], errors="coerce")
        if pd.api.types.is_numeric_dtype(table[column]) or numeric_values.notna().any():
            predictors.append(column)
    return predictors


available_context_columns = [
    column
    for column in ANALYSIS_CONTEXT_COLUMNS
    if column in result.analysis_table.columns
]
all_predictor_columns = infer_numeric_predictors(result.analysis_table)
shared_predictor_columns = [
    column
    for column in all_predictor_columns
    if not is_classification_only_feature(column)
]
classification_only_predictor_columns = [
    column for column in all_predictor_columns if is_classification_only_feature(column)
]
classification_predictor_columns = [
    *shared_predictor_columns,
    *classification_only_predictor_columns,
]

task_type = result.analysis_table["task_type"].astype("string").str.lower()
classification_mask = task_type.isin(CLASSIFICATION_PROBLEM_TYPES)

analysis_general_raw = result.analysis_table.loc[
    :, available_context_columns + shared_predictor_columns
].copy()
analysis_classification_raw = result.analysis_table.loc[
    classification_mask, available_context_columns + classification_predictor_columns
].copy()

feature_table_summary = pd.DataFrame(
    [
        {
            "table": "general_regression_and_classification",
            "rows": len(analysis_general_raw),
            "predictor_columns": len(shared_predictor_columns),
            "classification_only_predictors": 0,
        },
        {
            "table": "classification_augmented",
            "rows": len(analysis_classification_raw),
            "predictor_columns": len(classification_predictor_columns),
            "classification_only_predictors": len(
                classification_only_predictor_columns
            ),
        },
    ]
)

display(feature_table_summary)
print(
    f"Shared predictors retained before preprocessing: {len(shared_predictor_columns)}"
)
print(
    f"Classification-only predictors available before preprocessing: {len(classification_only_predictor_columns)}"
)

,table,rows,predictor_columns,classification_only_predictors
0,general_regression_and_classification,51,544,0
1,classification_augmented,38,1114,570


Shared predictors retained before preprocessing: 544
Classification-only predictors available before preprocessing: 570


## Preprocess Analysis Tables

For each analysis matrix, infinite values are treated as missing values before feature filtering. We then remove features with high missingness and features with negligible empirical variation. The near-constant filter removes exact constants, numerically constant features under a tight relative tolerance, and features where one rounded value dominates at least 95% of non-missing observations. These rules are unsupervised and are applied independently to the general and classification-specific matrices.

In [ ]:
MAX_FEATURE_MISSINGNESS = 0.20
NEAR_CONSTANT_TOP_SHARE = 0.95
NUMERICAL_CONSTANT_REL_TOL = 1e-12


def _near_constant_report(
    table: pd.DataFrame,
    feature_columns: list[str],
    *,
    top_share_threshold: float = NEAR_CONSTANT_TOP_SHARE,
    numerical_constant_rel_tol: float = NUMERICAL_CONSTANT_REL_TOL,
) -> pd.DataFrame:
    rows = []
    for column in feature_columns:
        values = pd.to_numeric(table[column], errors="coerce").dropna()
        n_non_missing = int(values.shape[0])
        if n_non_missing == 0:
            rows.append(
                {
                    "feature": column,
                    "n_non_missing": n_non_missing,
                    "n_unique": 0,
                    "top_share": np.nan,
                    "reason": "all_missing_after_inf_replacement",
                }
            )
            continue

        n_unique = int(values.nunique(dropna=True))
        rounded_values = values.round(12)
        top_share = float(
            rounded_values.value_counts(normalize=True, dropna=True).iloc[0]
        )
        scale = max(1.0, float(values.abs().median()))
        numerical_span = float(values.max() - values.min())

        reason = None
        if n_unique <= 1:
            reason = "exact_constant"
        elif numerical_span <= numerical_constant_rel_tol * scale:
            reason = "numerical_constant"
        elif top_share >= top_share_threshold:
            reason = "dominant_value_share_ge_0.95"

        if reason is not None:
            rows.append(
                {
                    "feature": column,
                    "n_non_missing": n_non_missing,
                    "n_unique": n_unique,
                    "top_share": top_share,
                    "reason": reason,
                }
            )
    return pd.DataFrame(rows)


def preprocess_analysis_table(
    table: pd.DataFrame,
    feature_columns: list[str],
    *,
    table_name: str,
    max_feature_missingness: float = MAX_FEATURE_MISSINGNESS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cleaned = table.copy()
    for column in cleaned.columns:
        numeric_values = pd.to_numeric(cleaned[column], errors="coerce")
        if (
            pd.api.types.is_numeric_dtype(cleaned[column])
            or numeric_values.notna().any()
        ):
            cleaned[column] = numeric_values.replace([np.inf, -np.inf], np.nan)
    feature_columns = [
        column for column in feature_columns if column in cleaned.columns
    ]

    missing_fraction = (
        cleaned[feature_columns].isna().mean().sort_values(ascending=False)
    )
    high_missing_features = missing_fraction[
        missing_fraction > max_feature_missingness
    ].index.tolist()
    after_missingness = [
        column for column in feature_columns if column not in high_missing_features
    ]

    near_constant = _near_constant_report(cleaned, after_missingness)
    near_constant_features = (
        near_constant["feature"].tolist() if not near_constant.empty else []
    )
    retained_features = [
        column for column in after_missingness if column not in near_constant_features
    ]

    context_columns = [
        column for column in available_context_columns if column in cleaned.columns
    ]
    processed = cleaned.loc[:, context_columns + retained_features].copy()

    report_rows = []
    for feature in high_missing_features:
        report_rows.append(
            {
                "table": table_name,
                "stage": "high_missingness",
                "feature": feature,
                "missing_fraction": float(missing_fraction.loc[feature]),
                "reason": f"> {max_feature_missingness:.0%} missing",
            }
        )
    if not near_constant.empty:
        near_constant = near_constant.assign(table=table_name, stage="near_constant")
        near_constant["missing_fraction"] = (
            near_constant["feature"].map(missing_fraction).astype(float)
        )
        report_rows.extend(
            near_constant[
                [
                    "table",
                    "stage",
                    "feature",
                    "missing_fraction",
                    "n_non_missing",
                    "n_unique",
                    "top_share",
                    "reason",
                ]
            ].to_dict("records")
        )

    report = pd.DataFrame(report_rows)
    print(
        f"{table_name}: rows={len(processed)}, predictors={len(retained_features)} "
        f"(dropped {len(high_missing_features)} high-missing, {len(near_constant_features)} near-constant)"
    )
    return processed, report


analysis_general, general_preprocessing_report = preprocess_analysis_table(
    analysis_general_raw,
    shared_predictor_columns,
    table_name="general_regression_and_classification",
)
analysis_classification, classification_preprocessing_report = (
    preprocess_analysis_table(
        analysis_classification_raw,
        classification_predictor_columns,
        table_name="classification_augmented",
    )
)

preprocessing_report = pd.concat(
    [general_preprocessing_report, classification_preprocessing_report],
    ignore_index=True,
)
preprocessing_summary = pd.DataFrame(
    [
        {
            "table": "general_regression_and_classification",
            "rows": len(analysis_general),
            "columns": analysis_general.shape[1],
            "predictors_after_preprocessing": analysis_general.shape[1]
            - len(available_context_columns),
        },
        {
            "table": "classification_augmented",
            "rows": len(analysis_classification),
            "columns": analysis_classification.shape[1],
            "predictors_after_preprocessing": analysis_classification.shape[1]
            - len(available_context_columns),
        },
    ]
)

display(preprocessing_summary)
display(preprocessing_report)

general_regression_and_classification: rows=51, predictors=488 (dropped 56 high-missing, 0 near-constant)
classification_augmented: rows=38, predictors=1022 (dropped 62 high-missing, 30 near-constant)


,table,rows,columns,predictors_after_preprocessing
0,general_regression_and_classification,51,507,488
1,classification_augmented,38,1041,1022


,table,stage,feature,missing_fraction,reason,n_non_missing,n_unique,top_share
0,general_regression_and_classification,high_missingness,pymfe__h_mean.skewness,0.470588,> 20% missing,NaN,NaN,NaN
1,general_regression_and_classification,high_missingness,pymfe__h_mean.kurtosis,0.470588,> 20% missing,NaN,NaN,NaN
2,general_regression_and_classification,high_missingness,pymfe__g_mean.kurtosis,0.470588,> 20% missing,NaN,NaN,NaN
3,general_regression_and_classification,high_missingness,pymfe__g_mean.skewness,0.470588,> 20% missing,NaN,NaN,NaN
4,general_regression_and_classification,high_missingness,pymfe__g_mean.histogram.6,0.352941,> 20% missing,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
143,classification_augmented,near_constant,pymfe__linear_discr.count,0.105263,dominant_value_share_ge_0.95,34.0,2.0,0.970588
144,classification_augmented,near_constant,pymfe__naive_bayes.count,0.105263,exact_constant,34.0,1.0,1.000000
145,classification_augmented,near_constant,pymfe__one_nn.count,0.105263,exact_constant,34.0,1.0,1.000000
146,classification_augmented,near_constant,pymfe__random_node.count,0.105263,exact_constant,34.0,1.0,1.000000
